# 1.4 · MCP、技能（Skills）与函数调用的区别

**MCP vs. Skills vs. Functions: who does what**

> **学完本课你能：** 用一句话分别说清函数调用、MCP 和智能体技能（Agent Skills），能针对具体需求选对方案，并展示三者如何在同一个智能体中叠加协作。

官方阅读材料：[Agent Skills specification](https://agentskills.io/specification) · [Skills extension](https://modelcontextprotocol.io/extensions/skills/overview) · [Build with Agent Skills](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-with-agent-skills) · [Build an MCP client (function calling loop)](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-client)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个任务，三个层级
我们会用三种方式实现同一个小任务——<em>“按照公司政策检查一笔费用”</em>——让差异变得具体可见。首先是<strong>函数调用（Function Calling）</strong>版本：一个 Python 函数，加上你会放进 Claude `tools=[...]` 列表中的工具定义（Tool Definition）。所有东西都在这个 notebook 里，它扮演“你的应用”。

In [ ]:
import json

POLICY = {"meals": 60, "hotel": 220, "taxi": 80}          # max EUR per item


def check_expense(category: str, amount: float) -> dict:
    """The business logic: is one expense within policy?"""
    limit = POLICY.get(category)
    if limit is None:
        return {"ok": False, "reason": f"unknown category {category!r}"}
    return {"ok": amount <= limit, "limit": limit, "over_by": max(0, amount - limit)}


# Function calling: YOU write the definition the model sees, by hand, inside your app.
FUNCTION_TOOL = {
    "name": "check_expense",
    "description": "Check one expense against company policy. Returns ok, the limit and how much it is over.",
    "input_schema": {
        "type": "object",
        "properties": {"category": {"type": "string", "enum": sorted(POLICY)},
                       "amount": {"type": "number", "description": "Amount in EUR"}},
        "required": ["category", "amount"],
    },
}

# What the model sends back when it wants the action (shape of a Claude `tool_use` block):
tool_use = {"type": "tool_use", "id": "toolu_01", "name": "check_expense",
            "input": {"category": "hotel", "amount": 260}}

# Your code executes it locally and returns a `tool_result` block:
result = check_expense(**tool_use["input"])
print(json.dumps({"type": "tool_result", "tool_use_id": tool_use["id"], "content": json.dumps(result)}, indent=2))

### 2. 同样的能力，做成 MCP 服务器
现在把逻辑移进一个 <strong>MCP 服务器（MCP Server）</strong>。宿主不再硬编码定义：它用 `tools/list` 来<em>发现</em>定义，再转换成模型的工具格式。注意转换后的定义与 `FUNCTION_TOOL` 的形状完全相同——<strong>MCP 是给函数调用供货，而不是取代它。</strong>

In [ ]:
from typing import Annotated, Literal

from pydantic import Field

from mcp import Client
from mcp.server import MCPServer

finance = MCPServer("finance")


@finance.tool()
def check_expense_tool(
    category: Annotated[Literal["hotel", "meals", "taxi"], Field(description="Expense category")],
    amount: Annotated[float, Field(description="Amount in EUR")],
) -> dict[str, object]:
    """Check one expense against company policy. Returns ok, the limit and how much it is over."""
    return check_expense(category, amount)


@finance.tool()
def list_expenses(employee: str) -> list[dict[str, object]]:
    """List last week's expenses for an employee."""
    return [{"id": 1, "category": "hotel", "amount": 300.0},
            {"id": 2, "category": "meals", "amount": 35.5},
            {"id": 3, "category": "taxi", "amount": 95.0}]


async with Client(finance) as client:
    mcp_tools = (await client.list_tools()).tools

# The host's one-line adapter: MCP tool -> model tool definition
as_model_tools = [{"name": t.name, "description": t.description, "input_schema": t.input_schema} for t in mcp_tools]
print(json.dumps(as_model_tools[0], indent=2))
print("\nsame keys as the hand-written function tool:", as_model_tools[0].keys() == FUNCTION_TOOL.keys())

### 3. 把专业知识做成智能体技能
工具说明的是<em>能做什么</em>。它们并没有说明<em>公司希望费用审查怎么做</em>：检查每一项、四舍五入到分、引用政策、超出 25 % 以上的一律升级。这套流程应该放进一个<strong>技能（Skill）</strong>：一个包含 `SKILL.md` 和一个参考文件的文件夹，写入一个临时目录。

In [ ]:
import tempfile
import textwrap
from pathlib import Path

skills_root = Path(tempfile.mkdtemp()) / "skills"
skill_dir = skills_root / "expense-review"
(skill_dir / "references").mkdir(parents=True)

(skill_dir / "SKILL.md").write_text(textwrap.dedent("""\
    ---
    name: expense-review
    description: Review an employee's expense report against company policy and flag violations. Use when asked to audit, check or approve expenses.
    ---
    # Expense review

    1. Call `list_expenses` for the employee.
    2. For EVERY item call `check_expense_tool` (never guess limits).
    3. If an item is more than 25% over its limit, mark it ESCALATE; otherwise mark it FLAG.
       The escalation rules are in references/escalation.md - read it only if something must be escalated.
    4. Finish with a table: id, category, amount, verdict.
"""))
(skill_dir / "references" / "escalation.md").write_text(
    "Escalations go to finance-approvals@example.com with the receipt attached, within 2 working days.\n")

for path in sorted(skill_dir.rglob("*")):
    print(path.relative_to(skills_root))

### 4. 一个带渐进式披露的迷你技能加载器
支持技能的宿主（Host）要做三件事：校验前置元数据（Frontmatter），只把<strong>元数据</strong>放进模型的上下文，以及<strong>按需</strong>加载正文或参考文件。观察每个层级的令牌估算值（≈ 字符数 ÷ 4）。

In [ ]:
import re

NAME_RULE = re.compile(r"^[a-z0-9]([a-z0-9-]{0,62}[a-z0-9])?$")   # 1-64 chars, a-z 0-9 -, no edge hyphens


def parse_skill(skill_md: Path) -> dict:
    text = skill_md.read_text()
    match = re.match(r"---\n(.*?)\n---\n(.*)", text, re.S)
    meta = dict(line.split(": ", 1) for line in match.group(1).splitlines())
    assert NAME_RULE.match(meta["name"]), f"invalid skill name {meta['name']!r}"
    assert 1 <= len(meta["description"]) <= 1024, "description must be 1-1024 characters"
    assert meta["name"] == skill_md.parent.name, "name should match the directory name"
    return {"name": meta["name"], "description": meta["description"], "body": match.group(2), "dir": skill_md.parent}


tokens = lambda s: len(s) // 4
catalog = {s["name"]: s for s in (parse_skill(p) for p in skills_root.glob("*/SKILL.md"))}

level1 = "\n".join(f"- {s['name']}: {s['description']}" for s in catalog.values())
print("LEVEL 1 - always in context (metadata):", tokens(level1), "tokens\n" + level1)
print("\nLEVEL 2 - on activation (SKILL.md body):", tokens(catalog["expense-review"]["body"]), "tokens")
ref = (catalog["expense-review"]["dir"] / "references" / "escalation.md").read_text()
print("LEVEL 3 - only if a step needs it (references/escalation.md):", tokens(ref), "tokens")
print("\nfor comparison, the MCP tool definitions sent with EVERY request:", tokens(json.dumps(as_model_tools)), "tokens")

### 5. 在一次智能体运行中用上全部三者
现在用一个智能体循环（Agent Loop）把所有东西串起来。一个脚本化的模型替身（Scripted Planner，因此可以离线运行）读取技能目录，激活 `expense-review`，通过发出 `tool_use` 调用来执行其中的步骤，宿主则通过 MCP 执行每一次调用。输出的跟踪信息会标出每一步由哪一层完成。

In [ ]:
def scripted_model(state: dict) -> dict:
    """A deterministic stand-in for the LLM that follows the skill's procedure step by step."""
    if "skill" not in state:
        return {"action": "activate_skill", "name": "expense-review"}            # chosen from level-1 metadata
    if "expenses" not in state:
        return {"action": "tool_use", "name": "list_expenses", "input": {"employee": "ana"}}
    pending = [e for e in state["expenses"] if e["id"] not in state["checked"]]
    if pending:
        e = pending[0]
        return {"action": "tool_use", "name": "check_expense_tool", "input": {"category": e["category"], "amount": e["amount"]}, "id": e["id"]}
    if any(v["over_by"] > 0.25 * v["limit"] for v in state["checked"].values()) and "escalation" not in state:
        return {"action": "read_skill_file", "path": "references/escalation.md"}
    return {"action": "final"}


state = {"checked": {}}
async with Client(finance) as client:
    while (step := scripted_model(state))["action"] != "final":
        if step["action"] == "activate_skill":
            state["skill"] = catalog[step["name"]]["body"]
            print(f"[SKILLS level 2]  load SKILL.md of {step['name']!r}")
        elif step["action"] == "read_skill_file":
            state["escalation"] = (catalog["expense-review"]["dir"] / step["path"]).read_text()
            print(f"[SKILLS level 3]  read {step['path']}")
        else:
            print(f"[FUNCTION CALL]   model emits tool_use {step['name']}({step['input']})")
            result = await client.call_tool(step["name"], step["input"])
            print(f"[MCP]             host sends tools/call to server 'finance' -> {result.structured_content}")
            if step["name"] == "list_expenses":
                state["expenses"] = result.structured_content["result"]   # lists are wrapped in {"result": ...}
            else:
                state["checked"][step["id"]] = result.structured_content     # a dict return is the object itself

print("\nid | category | amount | verdict")
for e in state["expenses"]:
    v = state["checked"][e["id"]]
    verdict = "OK" if v["ok"] else ("ESCALATE" if v["over_by"] > 0.25 * v["limit"] else "FLAG")
    print(f"{e['id']}  | {e['category']:<8} | {e['amount']:>6.2f} | {verdict}")

阅读这份跟踪信息：<strong>技能（Skill）</strong>决定了流程（检查每一项，只在需要时读取升级文件），<strong>函数调用（Function Calling）</strong>表达了每一个动作，<strong>MCP</strong> 则把每个动作送到拥有相应逻辑的服务器。

### 6. 可选：让真实模型来选择
设置了 `ANTHROPIC_API_KEY` 后，Claude 会在系统提示词中拿到技能目录、一个用于第 2 层级的 `load_skill` 函数，以及财务 MCP 工具——然后自行决定何时加载技能、调用哪些工具。没有密钥时，该单元格会说明本来会发生什么。

In [ ]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    load_skill = {"name": "load_skill", "description": "Load the full instructions of a skill by name.",
                  "input_schema": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}}
    system = "You can use these skills; call load_skill before following one.\n" + level1
    messages = [{"role": "user", "content": "Please audit ana's expenses from last week."}]
    async with Client(finance) as client:
        for _ in range(12):
            response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, system=system,
                                           tools=as_model_tools + [load_skill], messages=messages)
            if response.stop_reason != "tool_use":
                break
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                if block.name == "load_skill":
                    print("[SKILLS] Claude loaded", block.input)
                    content, is_error = catalog.get(block.input["name"], {}).get("body", "no such skill"), False
                else:
                    print("[MCP]    Claude called", block.name, block.input)
                    r = await client.call_tool(block.name, block.input)
                    content, is_error = json.dumps(r.structured_content), r.is_error
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": content, "is_error": is_error})
            messages.append({"role": "user", "content": results})
    print("\n".join(b.text for b in response.content if b.type == "text"))
else:
    print("skipped: set ANTHROPIC_API_KEY to watch Claude pick the skill from its description, "
          "load it, and then call the MCP tools in the order the skill prescribes.")

### 动手练习
1. 添加第二个技能 `travel-booking`，只包含前置元数据（Frontmatter）和三个步骤。重新运行第 4 节：第 1 层级增长了多少？`expense-review` 的第 2 层级有变化吗？
2. 把技能弄坏：在前置元数据中把它改名为 `Expense_Review`，然后重新运行第 4 节。违反了 Agent Skills 规范的哪条规则？
3. 把 25 % 的升级规则从 `SKILL.md` 移到 MCP 工具（Tool）里（改为返回一个 `verdict` 字段）。如果有三个不同的智能体使用这个服务器，但只有一家公司有这条规则，哪种设计更好？正反两方各用两句话论证。
4. 用新员工的类比，各用一句话向一位非技术背景的经理解释函数调用、MCP 和技能，然后计时，看自己能否在 30 秒内把三者都讲完。